# sim-app tutorial

This notebook introduces the public `sim_app` workflow:

1. Generate a small example dataset.
2. Summarize the returned `AnnData` object.
3. Plot aligned and unaligned coordinates.
4. Generate a custom simulation.
5. Save the result as `.h5ad`.

## Setup

If you have not installed the package yet, run this from the repository root in a terminal:

```bash
cd sim_app_package
python -m pip install -e ".[plot]"
```

The next cell also adds `sim_app_package/src` to `sys.path` when the notebook is run directly from this repository checkout.

In [ ]:
from pathlib import Path
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
for root in search_roots:
    src_path = root / "sim_app_package" / "src"
    if src_path.exists():
        if str(src_path) not in sys.path:
            sys.path.insert(0, str(src_path))
        break

import sim_app

sim_app.__version__

## Create a small example dataset

`example_data()` is meant for tutorials, smoke tests, and quick checks. It calls `generate_data()` with smaller defaults.

In [ ]:
adata = sim_app.example_data()
adata

## Summarize the object

`describe()` returns a plain dictionary with the main object metadata and available annotations.

In [ ]:
summary = sim_app.describe(adata)
summary

The returned object is a normal `AnnData` object. The count matrix is in `adata.X`, observation metadata is in `adata.obs`, gene metadata is in `adata.var`, and spatial coordinates are in `adata.obsm`.

In [ ]:
print("shape:", adata.shape)
print("obs columns:", list(adata.obs.columns))
print("var columns:", list(adata.var.columns))
print("coordinate keys:", list(adata.obsm.keys()))

## Plot the aligned 2D coordinates

The default example data returns bin observations cut along the Z axis. The aligned 2D view is stored in `adata.obsm["spatial"]`.

In [ ]:
fig = sim_app.plot(
    adata,
    view="2d",
    coordinates="aligned",
    color="domain_true",
    point_size=10,
)

## Plot unaligned coordinates

`coordinates="unaligned"` shows the deterministic per-slice rigid transform. This is useful for testing alignment workflows.

In [ ]:
fig = sim_app.plot(
    adata,
    view="2d",
    coordinates="unaligned",
    color="slice_id",
    point_size=10,
)

## Generate a custom dataset

`generate_data()` is the main entry point. Use keyword arguments to choose the output type, slicing axis, number of cells, and capture resolution.

Supported outputs are `"cell"`, `"bin"`, and `"spot"`. Supported slicing axes are `"X"`, `"Y"`, and `"Z"`; `"Z"` is the default.

In [ ]:
custom = sim_app.generate_data(
    output="spot",
    slice_axis="Y",
    n_cells=500,
    n_slices=4,
    sphere_radius_um=250.0,
    capture_window_um=(400.0, 400.0),
    spot_spacing_um=80.0,
    spot_radius_um=25.0,
    seed=7,
)

sim_app.describe(custom)

In [ ]:
fig = sim_app.plot(
    custom,
    view="2d",
    coordinates="aligned",
    color="cell_type_true",
    point_size=24,
)

## Save the generated data

`save()` writes the object in AnnData's `.h5ad` format and returns the path.

In [ ]:
output_dir = Path("tutorial_output")
output_dir.mkdir(exist_ok=True)

path = sim_app.save(custom, output_dir / "custom_spots_y.h5ad")
path

## Next steps

- Increase `n_cells` gradually for larger simulations.
- Use `output="bin"` with `bin_size_um` for VisiumHD-like observations.
- Use `output="spot"` with `spot_spacing_um` and `spot_radius_um` for Visium-like observations.
- Set `include_truth=False` when you want to hide ground-truth labels and composition annotations.